# Linear regression with multiple features/variables
Let's extend the idea of linear regression to work with multiple independent variables (Multivariate Linear Regression).

## Problem context:
We want to sell houses and based on some characteristics you want to know what a good market price would be.
We have gathered information on the recent houses that were sold in the area.

It is your job to predict housing prices based on these features.

The file housing_prices.txt contains a training set of housing prices in Portland, Oregon.

The data is organized as
- 1st column : size of the house (in square feet),
- 2nd column is the number of bedrooms,
- 3rd column is the price of the house.

The only difference with the previous example is that we now have more than one independent variables (but the concepts you have learnt in the previous section applies here as well).

In [82]:
# #################################################
# FILL IN THE NECESSARY CODE.
# Import the necessary packages
# ps: We need to be able to manipulate dataframes and perform matrix calculation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [83]:
# #################################################
# FILL IN THE NECESSARY CODE.
# Read the data from the file "ex1data2.txt" (available on Canvas) and display the top 5 rows of this data.
data = pd.read_csv(r'../house_pricing.txt', header = None, delimiter =  ",")
print(data.head(5))

      0  1       2
0  2104  3  399900
1  1600  3  329900
2  2400  3  369000
3  1416  2  232000
4  3000  4  539900


Fill in the following. Make sure that we have numpy arrays to continue our calculations

In [84]:
# #################################################
# FILL IN THE NECESSARY CODE.
# We need
#       X to hold all the features
#       y to hold the labels
#       m to hold the number of samples
X = data.iloc[:, 0:2]
print('X.shape: ', X.shape)
y = data.iloc[:, 2]
print('y.shape: ', y.shape)
m = len(y)
print('number of samples: ', m)
print(data.head())


X.shape:  (47, 2)
y.shape:  (47,)
number of samples:  47
      0  1       2
0  2104  3  399900
1  1600  3  329900
2  2400  3  369000
3  1416  2  232000
4  3000  4  539900


## Feature Normalization
Looking at the data, the house sizes are about 1000 times the number of bedrooms. 
So the house size is orders of magnitude larger, so the best is to "scale the features" or "normalize", 
this will have as benefit that gradient descent converge much more quickly.

We will
  - calculate the mean of each of the features (Xo, X1)
  - subtract the mean value of each feature from the dataset.
  - divide each feature values by their respective standard deviations

In [85]:
# #################################################
# FILL IN THE NECESSARY CODE.
# Normalize X
# Should we normalize y?
# Convert X into the variable X_norm
# Print out both the mean of X and the normalized output 

# axis=0 is for each column
mean = np.mean(X, axis=0)
stand = np.std(X, axis=0)
substraction = X - mean
X_norm = substraction/stand
print(f'mean: {mean}')
print(f'normalized output: {X_norm}')

mean: 0    2000.680851
1       3.170213
dtype: float64
normalized output:            0         1
0   0.131415 -0.226093
1  -0.509641 -0.226093
2   0.507909 -0.226093
3  -0.743677 -1.554392
4   1.271071  1.102205
5  -0.019945  1.102205
6  -0.593589 -0.226093
7  -0.729686 -0.226093
8  -0.789467 -0.226093
9  -0.644466 -0.226093
10 -0.077182  1.102205
11 -0.000866 -0.226093
12 -0.140779 -0.226093
13  3.150993  2.430504
14 -0.931924 -0.226093
15  0.380715  1.102205
16 -0.865783 -1.554392
17 -0.972626 -0.226093
18  0.773743  1.102205
19  1.310501  1.102205
20 -0.297227 -0.226093
21 -0.143323 -1.554392
22 -0.504553 -0.226093
23 -0.049200  1.102205
24  2.403094 -0.226093
25 -1.145609 -0.226093
26 -0.690256 -0.226093
27  0.668173 -0.226093
28  0.253521 -0.226093
29  0.809358 -0.226093
30 -0.205648 -1.554392
31 -1.272803 -2.882690
32  0.050011  1.102205
33  1.445326 -0.226093
34 -0.241262  1.102205
35 -0.716966 -0.226093
36 -0.968810 -0.226093
37  0.167030  1.102205
38  2.816474  1.102205
39  0.

Adding the intercept term and initializing parameters

In [86]:
# #################################################
# FILL IN THE NECESSARY CODE.
# Augment the matrix X so that we can perform a X*theta in 1 go. 
# So include the possibility to multiply in 1 go, including theta0.
# Convert X_norm into the variable X_aug
# So include the possibility to multiply X*theta in 1 go, including theta0 (the intercept).

# theta need a dimension of 3,1 because of the 2 features + bias
m = len(X_norm)
theta = np.zeros([3,1])

ones = np.ones((m, 1))
X_aug = np.hstack((ones, X_norm))
print('theta: ', theta.shape)
print('X.shape: ', X_aug.shape)

theta:  (3, 1)
X.shape:  (47, 3)


In [87]:
# #################################################
# FILL IN THE NECESSARY CODE.
# Initialize all the hyperparameters of our model (Learning rate, theta...)
num_iters = 10000
learning_rate = 0.03

## Computing the cost

In [88]:
# #################################################
# FILL IN THE NECESSARY CODE.
# Calculate the MSE.
y_reshaped = np.array(y).reshape(-1,1)
def computeCostMulti(X_aug, y, theta):
    m = len(y)
    local_cost = np.dot(X_aug, theta) - y_reshaped
    total_cost = np.sum(np.power(local_cost, 2)) / (2*m)
    return total_cost

In [89]:
cost = computeCostMulti(X_aug, y, theta)
print(cost)

65591548106.45744


<b>Check your calculation</b>

Cost :  65591548106.45744

## Creating the optimizer: Gradient Descent

In [90]:
# #################################################
# FILL IN THE NECESSARY CODE.
#    X contains the multivaraite input
#    y contain the labels
#    theta : parameters list of our linear model
#    learning_rate : the learning rate
#    num_iters : number of iterations
#    return parameter (theta) : parameters list of our linear model

def gradientDescentMulti(X_aug, y, theta, learning_rate, num_iters):
    for _ in range(num_iters):
        y_pred = X_aug.dot(theta)
        error = y_pred - y
        gradient = 2*X_aug.T.dot(error)/len(X_aug)
        theta -= learning_rate*gradient
    return theta

In [91]:
theta_refined = gradientDescentMulti(X_aug, y_reshaped, theta, learning_rate, num_iters)
print(theta)

[[340412.65957447]
 [109447.79646964]
 [ -6578.35485416]]


<pre>
(results with learning_rate = 0.03)
Theta with    100 iterations : [[3.39642905e+05]  [1.44952956e+05]  [5.98800848e+01]]
Theta with    200 iterations : [[3.39642905e+05]  [1.44952956e+05]  [5.98800848e+01]]
Theta with    400 iterations : [[3.40410919e+05]  [1.53263978e+05]  [4.64918053e+01]]
Theta with   1000 iterations : [[3.40412660e+05]  [1.53769414e+05]  [-6.77111420e+00]]
Theta with   5000 iterations : [[340412.65957447] [153769.70114155] [-363.65654995]]
Theta with  10000 iterations : [[340412.65957447] [153769.94033767] [-809.74547603]]
</pre>
We now have the optimized value of theta. Use these values in the above cost function.

In [92]:
cost_multi = computeCostMulti(X_aug, y, theta)
print(cost_multi)

2043280050.6028287


<pre>
Cost with    100 iterations :   10596969344.166979
Cost with    200 iterations :   3344770635.4916563
Cost with    400 iterations :   2105448288.6292474
Cost with   1000 iterations :   2043498948.1433077
Cost with   5000 iterations :   2043280050.6028287
Cost with  10000 iterations :   2043280050.6028283
</pre>

## Predicting
What would be the price for a 2480 square foot, 3 bedroom house?

How confident are you about this prediction?

In [93]:
# #################################################
# FILL IN THE NECESSARY CODE.
value_to_be_predicted = [2480, 3]
new_X = np.array([value_to_be_predicted])
# normalize the new value
mean = np.mean(X, axis=0).values
stand = np.std(X, axis=0).values
new_X_norm = (new_X - mean) / stand
new_X_b = np.c_[np.ones((1, 1)), new_X_norm]
predicted_y = new_X_b.dot(theta_refined)
print(f'Predicted y for X={value_to_be_predicted}: ${predicted_y[0,0]:.2f}')

Predicted y for X=[2480, 3]: $408626.32


What would be the price for a 5500 square foot, 6 bedroom house?

How confident are you about this prediction?

In [94]:
# #################################################
# FILL IN THE NECESSARY CODE.
value_to_be_predicted = [5500, 6]
new_X = np.array([value_to_be_predicted])
# normalize the new value
mean = np.mean(X, axis=0).values
stand = np.std(X, axis=0).values
new_X_norm = (new_X - mean) / stand
new_X_b = np.c_[np.ones((1,1)), new_X_norm]
predicted_y = new_X_b.dot(theta_refined)
print(f'Predicted y for X={value_to_be_predicted}: ${predicted_y[0,0]:.2f}')

Predicted y for X=[5500, 6]: $802828.50


# Try now using Scikit-learn


(If you prefer, create a new notebook)